# 记忆块和闲时计算（Letta）

MemGPT在2024年变成了Letta。2026年进一步进化：功能离散，模型可直接编辑的记忆块，以及一个闲时agent在主agent闲置时异步整合记忆。

## 问题描述

MemGPT解决了虚拟记忆控制流，生产环境下三个问题涌现：
- 延迟。  所有的记忆操作都坐落在主路径。如果agent在用户等待的时候触发了剪枝、总结或者调和，尾延迟就会爆炸。
- 记忆腐烂。  不断累积的写，过时的事实被保留。检索被过时的内容淹没。
- 结构损失。  扁平的档案存储不能够表达“Human块始终在prompt中，Persona块始终在prompt中，Task块会话间切换”。

Letta 2026年的重构，记忆块让结构显式，闲时计算将记忆整合移除出关键路径。

## 基本概念

### 三层

|层|可见性|在哪里有效|由谁写|
|---|---|---|---|
|Core|始终可见|在主prompt中|Agent工具调用或者闲时重写|
|Recall|会话历史|可检索|会话轮次自动登记|
|Archival|任意事实|向量+KV+图|Agent工具调用或者闲时摄入|

Core 就是MemGPT的Core，Recall是驱逐尾部的会话缓存，Archival是外部数据库。

### 记忆块

一个块是带类型的、持久的、可编辑的核心层会话。原始的MemGPT论文定义了两种：
- Human块 ————关于用户的事实（名字、角色、偏好、目标）
- Persona块 ————Agent的自我概念（身份、语气、约束）

Letta将其泛化到任意用户定义的块：一个`Task`块面向当前目标，一个`Project`块面向代码库事实，一个`Safety`块面向硬性约束。每个块都有`id`、`label`、`value`、`limit`、`description`。

块编辑的工具接口
```
block_append(label, text)
block_replace(label, old, new)
block_read(label)
block_summarize(label)
```

### 闲时计算

Letta 2025年做的事：在后台跑第二个agent，脱离关键路径。闲时agent负责处理会话抄录、代码库上下文...

好处：
- 没有延迟开销。   主响应不需要等待记忆操作。
- 能用更强的模型。  闲时agent可以用更贵、更强的模型，因为不受延迟约束。
- 天然的整合窗口。  在用户不等待时去重、总结、失效被推翻的事实。

行为就像人类工作一样： 做完工作、然后去睡觉，在此期间形成长期记忆。

### 什么时候这种模式失败

- 块膨胀。  无限的`block_append`导致快速触碰上限。需要进行压缩总结。
- 静默漂移。  显示agent把一个块重写了，但是主agent没有感知到。给块加版本，在轨迹里暴露diff。
- 整合投毒。  闲时agent把攻击者可触碰到的内容塞进了core。

# 开始编码

对应本章核心：**三层记忆（Core 块 / Recall / Archival）**、**可编辑记忆块（limit + version）**、**闲时计算（整合移出关键路径）**。  
先用脚本化玩具跑通块工具与 sleep consolidate；再用 **PyTorch** 学「该不该 summarize / 是否过时」；最后用 **LangChain + DeepSeek** 跑主 agent + 闲时 agent。


## 1. 教学玩具：记忆块 + 闲时整合骨架

- **Core**：带 `limit` / `version` 的 typed blocks（human / persona / task…）。
- **Recall**：会话抄录（可检索）；**Archival**：外部事实库。
- **在线路径**：只做轻量 `block_*`；重整合交给 **sleep-time**，主回复不等待。


In [ ]:
from __future__ import annotations

import re
import time
import uuid
from dataclasses import dataclass, field
from difflib import unified_diff
from typing import Any, Callable

from typing_extensions import TypedDict


@dataclass
class MemoryBlock:
    """Letta 风格记忆块：持久、可编辑、有容量上限。"""

    label: str
    description: str
    value: str = ""
    limit: int = 120
    block_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    version: int = 1

    def fill_ratio(self) -> float:
        """
        Returns:
            ratio: 相对 ``limit`` 的占用比例。
        """
        return len(self.value) / max(self.limit, 1)

    def render(self) -> str:
        """
        Returns:
            text: 注入主 prompt 的块视图（含 version，防静默漂移）。
        """
        body = self.value or "<empty>"
        return (
            f"### [{self.label}] id={self.block_id} v{self.version} "
            f"({len(self.value)}/{self.limit})\n"
            f"{self.description}\n{body}"
        )


@dataclass
class BlockDiff:
    """一次块写入的版本 diff（暴露给轨迹）。"""

    label: str
    old_version: int
    new_version: int
    patch: str


@dataclass
class BlockStore:
    """Core 层：按 label 管理记忆块。"""

    blocks: dict[str, MemoryBlock] = field(default_factory=dict)
    diffs: list[BlockDiff] = field(default_factory=list)

    def ensure(
        self,
        label: str,
        *,
        description: str = "",
        limit: int = 120,
    ) -> MemoryBlock:
        """
        Args:
            label: 块名。
            description: 用途说明。
            limit: 字符上限。

        Returns:
            block: 已有或新建块。
        """
        if label not in self.blocks:
            self.blocks[label] = MemoryBlock(
                label=label, description=description or f"{label} block", limit=limit
            )
        return self.blocks[label]

    def _commit(self, block: MemoryBlock, old_value: str) -> BlockDiff:
        """
        Args:
            block: 已改写的块（version 将 +1）。
            old_value: 改前文本。

        Returns:
            diff: 版本与 unified diff。
        """
        old_v = block.version
        block.version += 1
        patch = "\n".join(
            unified_diff(
                old_value.splitlines(),
                block.value.splitlines(),
                fromfile=f"{block.label}@v{old_v}",
                tofile=f"{block.label}@v{block.version}",
                lineterm="",
            )
        )
        diff = BlockDiff(block.label, old_v, block.version, patch or "(no textual diff)")
        self.diffs.append(diff)
        return diff

    def append(self, label: str, text: str) -> str:
        """
        Args:
            label: 块名。
            text: 追加文本。

        Returns:
            observation: 结果；触顶时提示 summarize。
        """
        block = self.ensure(label)
        old = block.value
        sep = "" if not block.value else " | "
        candidate = (block.value + sep + text).strip()
        if len(candidate) > block.limit:
            return (
                f"Error: block[{label}] would exceed limit "
                f"{block.limit} (need block_summarize first)"
            )
        block.value = candidate
        diff = self._commit(block, old)
        return f"OK: append {label} -> v{diff.new_version} fill={block.fill_ratio():.2f}"

    def replace(self, label: str, old: str, new: str) -> str:
        """
        Args:
            label: 块名。
            old: 旧子串。
            new: 新子串。

        Returns:
            observation: 成功或错误。
        """
        block = self.ensure(label)
        if old not in block.value:
            return f"Error: old text not found in block[{label}]"
        prev = block.value
        candidate = block.value.replace(old, new, 1)
        if len(candidate) > block.limit:
            return f"Error: replace would exceed limit on block[{label}]"
        block.value = candidate
        diff = self._commit(block, prev)
        return f"OK: replace {label} -> v{diff.new_version}"

    def read(self, label: str) -> str:
        """
        Args:
            label: 块名。

        Returns:
            observation: 块全文视图。
        """
        if label not in self.blocks:
            return f"Error: unknown block {label}"
        return self.blocks[label].render()

    def summarize(self, label: str, summary: str | None = None) -> str:
        """
        压缩块内容到 limit 内（玩具：保留关键词句或外部 summary）。

        Args:
            label: 块名。
            summary: 可选显式摘要；否则启发式截断去重。

        Returns:
            observation: 压缩结果。
        """
        block = self.ensure(label)
        old = block.value
        if summary is not None:
            compressed = summary.strip()
        else:
            parts = [p.strip() for p in re.split(r"\s*\|\s*", block.value) if p.strip()]
            # 去重保序
            seen: set[str] = set()
            uniq: list[str] = []
            for p in parts:
                key = p.lower()
                if key in seen:
                    continue
                seen.add(key)
                uniq.append(p)
            compressed = " | ".join(uniq)
            while len(compressed) > block.limit and uniq:
                uniq.pop(0)
                compressed = " | ".join(uniq)
            if len(compressed) > block.limit:
                compressed = compressed[: block.limit - 3] + "..."
        if len(compressed) > block.limit:
            return f"Error: summary still exceeds limit on block[{label}]"
        block.value = compressed
        diff = self._commit(block, old)
        return f"OK: summarize {label} -> v{diff.new_version} fill={block.fill_ratio():.2f}"

    def render_core(self) -> str:
        """
        Returns:
            prompt: 始终可见的 Core 层。
        """
        lines = ["# Core memory blocks (always in prompt)"]
        for label in sorted(self.blocks):
            lines.append(self.blocks[label].render())
            lines.append("")
        return "\n".join(lines).rstrip()


@dataclass
class RecallTurn:
    """Recall 层一条会话记录。"""

    turn_id: int
    role: str
    content: str


@dataclass
class RecallLog:
    """会话历史缓存（可检索；尾部可驱逐示意）。"""

    turns: list[RecallTurn] = field(default_factory=list)
    _next: int = 0

    def add(self, role: str, content: str) -> int:
        """
        Args:
            role: ``user`` / ``assistant`` / ``tool`` / ``sleep``.
            content: 文本。

        Returns:
            turn_id: 新轮次号。
        """
        self._next += 1
        self.turns.append(RecallTurn(self._next, role, content))
        return self._next

    def search(self, query: str, top_k: int = 3) -> str:
        """
        Args:
            query: 查询。
            top_k: 条数。

        Returns:
            observation: 命中列表。
        """
        q = set(re.findall(r"\w+", query.lower()))
        scored: list[tuple[float, RecallTurn]] = []
        for t in self.turns:
            toks = set(re.findall(r"\w+", t.content.lower()))
            score = len(q & toks) / max(len(q), 1)
            if score > 0:
                scored.append((score, t))
        scored.sort(key=lambda x: x[0], reverse=True)
        hits = scored[:top_k]
        if not hits:
            return "Observation: recall miss"
        lines = ["Observation: recall hits"]
        for score, t in hits:
            lines.append(f"- turn={t.turn_id} role={t.role} score={score:.2f}: {t.content}")
        return "\n".join(lines)

    def transcript(self, max_turns: int | None = None) -> str:
        """
        Args:
            max_turns: 可选截断。

        Returns:
            text: 闲时 agent 可读抄录。
        """
        turns = self.turns if max_turns is None else self.turns[-max_turns:]
        return "\n".join(f"[{t.turn_id}] {t.role}: {t.content}" for t in turns)


@dataclass
class ArchivalStore:
    """Archival 层：外部事实（玩具词重叠检索）。"""

    entries: list[str] = field(default_factory=list)

    def insert(self, text: str) -> str:
        """
        Args:
            text: 事实文本。

        Returns:
            observation: 确认。
        """
        self.entries.append(text)
        return f"OK: archival insert n={len(self.entries)}"

    def search(self, query: str, top_k: int = 3) -> str:
        """
        Args:
            query: 查询。
            top_k: 条数。

        Returns:
            observation: 命中。
        """
        q = set(re.findall(r"\w+", query.lower()))
        scored: list[tuple[float, str]] = []
        for e in self.entries:
            toks = set(re.findall(r"\w+", e.lower()))
            score = len(q & toks) / max(len(q), 1)
            if score > 0:
                scored.append((score, e))
        scored.sort(key=lambda x: x[0], reverse=True)
        hits = scored[:top_k]
        if not hits:
            return "Observation: archival miss"
        lines = ["Observation: archival hits"]
        for score, e in hits:
            lines.append(f"- score={score:.2f}: {e}")
        return "\n".join(lines)


@dataclass
class LettaState:
    """三层记忆状态。"""

    core: BlockStore = field(default_factory=BlockStore)
    recall: RecallLog = field(default_factory=RecallLog)
    archival: ArchivalStore = field(default_factory=ArchivalStore)

    def __post_init__(self) -> None:
        self.core.ensure("persona", description="Agent self-concept", limit=100)
        self.core.ensure("human", description="Facts about the user", limit=100)
        self.core.ensure("task", description="Current goal", limit=80)
        self.core.blocks["persona"].value = "Helpful coding assistant."
        # persona 初始写入不计演示 diff 噪声：清空 diffs
        self.core.diffs.clear()
        self.core.blocks["persona"].version = 1


class BlockTools:
    """四件块编辑工具（在线路径可用）。"""

    def __init__(self, state: LettaState) -> None:
        self.state = state

    def block_append(self, label: str, text: str) -> str:
        """追加到指定块。"""
        return self.state.core.append(label, text)

    def block_replace(self, label: str, old: str, new: str) -> str:
        """替换块内子串。"""
        return self.state.core.replace(label, old, new)

    def block_read(self, label: str) -> str:
        """读取块。"""
        return self.state.core.read(label)

    def block_summarize(self, label: str, summary: str | None = None) -> str:
        """压缩块。"""
        return self.state.core.summarize(label, summary=summary)

    def as_registry(self) -> dict[str, Callable[..., str]]:
        """
        Returns:
            tools: 工具名 → 可调用。
        """
        return {
            "block_append": self.block_append,
            "block_replace": self.block_replace,
            "block_read": self.block_read,
            "block_summarize": self.block_summarize,
        }


@dataclass
class SleepResult:
    """一次闲时整合的结果。"""

    elapsed_ms: float
    actions: list[str]
    diffs: list[BlockDiff]


class SleepTimeConsolidator:
    """
    闲时 agent（玩具）：扫 recall → 去重写入块 / 失效旧事实 / 必要时 summarize。
    故意设计为可在主回复返回之后再跑（不阻塞在线延迟）。
    """

    def __init__(self, state: LettaState) -> None:
        self.state = state
        self.tools = BlockTools(state)

    def consolidate(self) -> SleepResult:
        """
        Returns:
            result: 动作列表与块 diff。
        """
        t0 = time.perf_counter()
        before = len(self.state.core.diffs)
        actions: list[str] = []
        transcript = self.state.recall.transcript()

        # 1) 从抄录抽取偏好 / 任务（规则玩具）
        prefs = re.findall(r"prefer[s]?\s+(\w+)", transcript, flags=re.I)
        if prefs:
            fact = f"prefers {prefs[-1]}"
            human = self.state.core.blocks["human"].value.lower()
            if fact.lower() not in human:
                # 失效旧 prefers
                old_prefs = re.findall(r"prefers\s+\w+", self.state.core.blocks["human"].value, flags=re.I)
                for op in old_prefs:
                    self.tools.block_replace("human", op, fact)
                    actions.append(f"invalidate:{op}->{fact}")
                    break
                else:
                    obs = self.tools.block_append("human", fact)
                    if obs.startswith("Error"):
                        self.tools.block_summarize("human")
                        actions.append("summarize:human")
                        obs = self.tools.block_append("human", fact)
                    actions.append(f"append:human:{fact}|{obs}")

        tasks = re.findall(r"(?:goal|task)\s*[:=]\s*([^。.\n|]+)", transcript, flags=re.I)
        if tasks:
            goal = tasks[-1].strip()
            cur = self.state.core.blocks["task"].value
            if goal and goal not in cur:
                if cur:
                    self.tools.block_replace("task", cur, goal)
                    actions.append(f"replace:task->{goal}")
                else:
                    self.tools.block_append("task", goal)
                    actions.append(f"append:task:{goal}")

        # 2) 块膨胀：fill>0.85 则 summarize
        for label, block in list(self.state.core.blocks.items()):
            if block.fill_ratio() > 0.85:
                self.tools.block_summarize(label)
                actions.append(f"summarize:{label}:fill={block.fill_ratio():.2f}")

        # 3) 有价值事实进 archival
        for m in re.finditer(r"fact:\s*([^。.\n|]+)", transcript, flags=re.I):
            text = m.group(1).strip()
            if text and text not in self.state.archival.entries:
                self.state.archival.insert(text)
                actions.append(f"archival:{text}")

        new_diffs = self.state.core.diffs[before:]
        elapsed = (time.perf_counter() - t0) * 1000
        note = (
            f"[sleep] consolidated actions={actions} "
            f"diffs={[d.label + f'@v{d.new_version}' for d in new_diffs]}"
        )
        self.state.recall.add("sleep", note)
        return SleepResult(elapsed_ms=elapsed, actions=actions, diffs=new_diffs)


@dataclass
class ToolCall:
    """一次工具调用。"""

    name: str
    args: dict[str, Any]


class LLMReply(TypedDict, total=False):
    """脚本化 LLM 一步。"""

    kind: str  # action | finish
    thought: str
    action: str
    args: dict[str, Any]
    content: str


@dataclass
class OnlineAgent:
    """在线主路径：轻量块工具；不在此做重整合。"""

    state: LettaState
    tools: BlockTools
    max_steps: int = 6

    def dispatch(self, call: ToolCall) -> str:
        """
        Args:
            call: 工具调用。

        Returns:
            observation: 结果。
        """
        fn = self.tools.as_registry().get(call.name)
        if fn is None:
            return f"Error: unknown tool {call.name}"
        try:
            return fn(**call.args)
        except Exception as e:  # noqa: BLE001
            return f"Error: {type(e).__name__}: {e}"

    def run_scripted(self, user_text: str, script: list[LLMReply]) -> tuple[str, float]:
        """
        Args:
            user_text: 用户输入。
            script: 脚本化步骤。

        Returns:
            final: 助手回复。
            online_ms: 在线路径耗时（不含 sleep）。
        """
        t0 = time.perf_counter()
        self.state.recall.add("user", user_text)
        final = ""
        for step, reply in enumerate(script):
            if step >= self.max_steps:
                break
            if reply.get("kind", "finish") == "action":
                call = ToolCall(str(reply["action"]), dict(reply.get("args") or {}))
                obs = self.dispatch(call)
                self.state.recall.add("tool", f"[{call.name}] {obs}")
                continue
            final = str(reply.get("content", ""))
            self.state.recall.add("assistant", final)
            break
        return final, (time.perf_counter() - t0) * 1000


print(
    "Letta toy ready | blocks =",
    list(LettaState().core.blocks),
    "| tools =",
    list(BlockTools(LettaState()).as_registry()),
)


## 2. 玩具示例：在线轻写 → 触顶 → 闲时整合（主路径不阻塞）


In [ ]:
def demo_blocks_and_sleep() -> None:
    """
    1) 在线多次 append，第三次触顶失败；
    2) 先返回用户答案（低延迟）；
    3) 再跑 sleep consolidate：summarize + 抽取 prefers/task + archival；
    4) version/diff 可见。
    """
    state = LettaState()
    # 小 limit 便于演示膨胀
    state.core.blocks["human"].limit = 40
    tools = BlockTools(state)
    online = OnlineAgent(state=state, tools=tools)
    sleep = SleepTimeConsolidator(state)

    # --- 在线：尽量写，但不做重整合 ---
    script: list[LLMReply] = [
        {
            "kind": "action",
            "action": "block_append",
            "args": {"label": "human", "text": "name=Ada"},
        },
        {
            "kind": "action",
            "action": "block_append",
            "args": {"label": "human", "text": "role=PM"},
        },
        {
            "kind": "action",
            "action": "block_append",
            "args": {"label": "human", "text": "prefers bullets and long explanations forever"},
        },
        {
            "kind": "finish",
            "content": "记下了。我稍后再整理记忆。",
        },
    ]
    final, online_ms = online.run_scripted(
        "我是 Ada，PM。goal: ship MemBlock demo。fact: repo uses DeepSeek. prefers concise",
        script,
    )
    assert "记下" in final
    # 第三次 append 应因 limit 失败（长句）
    tool_obs = [t.content for t in state.recall.turns if t.role == "tool"]
    assert any("exceed limit" in o for o in tool_obs)

    print("=== online path (no sleep yet) ===")
    print(f"reply: {final}")
    print(f"online_ms: {online_ms:.3f}")
    print(state.core.render_core())

    # --- 闲时：用户已拿到回复后再整合 ---
    result = sleep.consolidate()
    print("\n=== sleep-time consolidate ===")
    print(f"sleep_ms: {result.elapsed_ms:.3f}")
    print("actions:", result.actions)
    for d in result.diffs:
        print(f"diff {d.label} v{d.old_version}->v{d.new_version}")
        print(d.patch)

    human = state.core.blocks["human"]
    assert human.version >= 2
    assert "prefers" in human.value.lower() or len(human.value) <= human.limit
    assert human.fill_ratio() <= 1.0
    assert any("MemBlock" in a or "task" in a for a in result.actions) or state.core.blocks["task"].value
    assert any("DeepSeek" in e for e in state.archival.entries)

    # 在线耗时应与 sleep 解耦：sleep 发生在返回之后
    assert result.elapsed_ms >= 0
    print("\n=== after sleep ===")
    print(state.core.render_core())
    print("archival:", state.archival.entries)
    print("TOY DEMO OK")


demo_blocks_and_sleep()


## 3. PyTorch：块膨胀 / 事实过时 分类头

用简单特征训练两个小 MLP：是否该 ``block_summarize``、某条事实是否过时（闲时失效信号）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def block_features(
    fill_ratio: float,
    n_segments: int,
    n_duplicates: int,
    chars: int,
) -> torch.Tensor:
    """
    Args:
        fill_ratio: 占用比例。
        n_segments: ``|`` 分段数。
        n_duplicates: 重复段数。
        chars: 当前字符数。

    Returns:
        x: ``(4,)`` 特征。
    """
    return torch.tensor(
        [fill_ratio, float(n_segments), float(n_duplicates), float(chars) / 100.0],
        dtype=torch.float32,
    )


def fact_features(
    age_turns: int,
    contradicted: bool,
    retrieval_hits: int,
    confidence: float,
) -> torch.Tensor:
    """
    Args:
        age_turns: 事实写入后经过的轮次。
        contradicted: 是否被更新偏好推翻。
        retrieval_hits: 近期检索命中次数。
        confidence: 写入置信度。

    Returns:
        x: ``(4,)`` 特征。
    """
    return torch.tensor(
        [float(age_turns), 1.0 if contradicted else 0.0, float(retrieval_hits), confidence],
        dtype=torch.float32,
    )


class BinaryHead(nn.Module):
    """四维特征 → logits。"""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(4,)`` 或 ``(B, 4)``。

        Returns:
            logits: ``(2,)`` 或 ``(B, 2)``。
        """
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        logits = self.net(x)
        return logits.squeeze(0) if single else logits


def train_binary(
    model: BinaryHead,
    data: list[tuple[torch.Tensor, int]],
    *,
    steps: int = 300,
    lr: float = 0.05,
) -> BinaryHead:
    """
    Args:
        model: 分类头。
        data: ``(features, label)``，label∈{0,1}。
        steps: 步数。
        lr: 学习率。

    Returns:
        model: 训练后模型。
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for x, y in data:
            logits = model(x)
            loss = loss + F.cross_entropy(logits.unsqueeze(0), torch.tensor([y]))
        loss = loss / max(len(data), 1)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


@torch.no_grad()
def predict_positive(model: BinaryHead, x: torch.Tensor) -> bool:
    """
    Args:
        model: 二分类头。
        x: 特征。

    Returns:
        positive: 预测为类别 1。
    """
    return int(torch.argmax(model(x)).item()) == 1


def demo_pytorch_memory_policy() -> None:
    """训练 summarize / stale 两个头并做断言。"""
    torch.manual_seed(0)

    summarize_data = [
        (block_features(0.95, 6, 2, 95), 1),
        (block_features(0.90, 5, 1, 90), 1),
        (block_features(0.40, 2, 0, 40), 0),
        (block_features(0.20, 1, 0, 20), 0),
        (block_features(0.88, 4, 3, 88), 1),
        (block_features(0.50, 3, 0, 50), 0),
    ]
    stale_data = [
        (fact_features(20, True, 0, 0.4), 1),
        (fact_features(15, True, 1, 0.5), 1),
        (fact_features(2, False, 5, 0.9), 0),
        (fact_features(1, False, 3, 0.8), 0),
        (fact_features(30, False, 0, 0.3), 1),
        (fact_features(5, False, 4, 0.7), 0),
    ]

    summarize_net = train_binary(BinaryHead(), summarize_data)
    stale_net = train_binary(BinaryHead(), stale_data)

    print("=== should summarize? ===")
    bloated = block_features(0.93, 5, 2, 93)
    slim = block_features(0.25, 1, 0, 25)
    print("bloated", predict_positive(summarize_net, bloated))
    print("slim", predict_positive(summarize_net, slim))
    assert predict_positive(summarize_net, bloated)
    assert not predict_positive(summarize_net, slim)

    print("\n=== is stale? ===")
    old = fact_features(25, True, 0, 0.4)
    fresh = fact_features(2, False, 4, 0.85)
    print("old contradicted", predict_positive(stale_net, old))
    print("fresh", predict_positive(stale_net, fresh))
    assert predict_positive(stale_net, old)
    assert not predict_positive(stale_net, fresh)
    print("PYTORCH DEMO OK")


demo_pytorch_memory_policy()


## 4. 生产级：LangChain 主 Agent + 闲时 Agent（DeepSeek）

主路径：``create_agent`` + 四件 ``block_*`` 工具，尽快回复。  
闲时路径：第二个 agent 读 recall 抄录，调用块工具做去重/总结/失效；``version``/diff 写入轨迹。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

PROD_STATE = LettaState()
PROD_TOOLS = BlockTools(PROD_STATE)


class AppendArgs(BaseModel):
    """block_append。"""

    label: str = Field(description="Block label: human|persona|task|...")
    text: str = Field(description="Text to append")


class ReplaceArgs(BaseModel):
    """block_replace。"""

    label: str = Field(description="Block label")
    old: str = Field(description="Exact old substring")
    new: str = Field(description="Replacement")


class ReadArgs(BaseModel):
    """block_read。"""

    label: str = Field(description="Block label")


class SummarizeArgs(BaseModel):
    """block_summarize。"""

    label: str = Field(description="Block label")
    summary: str | None = Field(default=None, description="Optional explicit summary")


def _make_tool(name: str, description: str, args_model: type[BaseModel], method: str) -> StructuredTool:
    """
    Args:
        name: 工具名。
        description: 说明。
        args_model: Pydantic 参数。
        method: ``BlockTools`` 方法名。

    Returns:
        tool: StructuredTool。
    """

    def _run(**kwargs: Any) -> str:
        parsed = args_model(**kwargs)
        data = parsed.model_dump()
        return str(getattr(PROD_TOOLS, method)(**data))

    return StructuredTool.from_function(
        name=name,
        description=description,
        func=_run,
        args_schema=args_model,
    )


def build_block_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 四件块工具。
    """
    return [
        _make_tool("block_append", "Append text to a core memory block.", AppendArgs, "block_append"),
        _make_tool("block_replace", "Replace substring in a core memory block.", ReplaceArgs, "block_replace"),
        _make_tool("block_read", "Read a core memory block.", ReadArgs, "block_read"),
        _make_tool(
            "block_summarize",
            "Compress a block under its limit; pass summary when possible.",
            SummarizeArgs,
            "block_summarize",
        ),
    ]


BLOCK_LC_TOOLS = build_block_tools()


def reset_prod_state() -> None:
    """重置生产全局状态与工具。"""
    global PROD_STATE, PROD_TOOLS, BLOCK_LC_TOOLS
    PROD_STATE = LettaState()
    PROD_STATE.core.blocks["human"].limit = 80
    PROD_TOOLS = BlockTools(PROD_STATE)
    BLOCK_LC_TOOLS = build_block_tools()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def format_agent_messages(messages: list[BaseMessage]) -> str:
    """
    Args:
        messages: 轨迹。

    Returns:
        text: 可读摘要。
    """
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args')})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"OBS[{m.name}]: {m.content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    """
    Args:
        messages: 消息。

    Returns:
        n: 工具调用次数。
    """
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


def build_online_agent() -> Any:
    """
    Returns:
        agent: 在线主 agent（轻量记忆写）。
    """
    system = (
        "You are the ONLINE Letta-style agent.\n"
        "Core memory blocks are always visible below.\n"
        "On lasting user facts, you MAY call block_append / block_replace lightly.\n"
        "If a block hits its limit, you may block_summarize, but prefer to answer the user quickly.\n"
        "Do NOT do heavy multi-step memory reconciliation online — sleep-time will handle it.\n"
        "Reply in Chinese, concise."
    )
    return create_agent(get_llm(), BLOCK_LC_TOOLS, system_prompt=system)


def build_sleep_agent() -> Any:
    """
    Returns:
        agent: 闲时整合 agent。
    """
    system = (
        "You are the SLEEP-TIME memory consolidator.\n"
        "Read the transcript and current blocks. Your job:\n"
        "1) Deduplicate and compress bloated blocks via block_summarize.\n"
        "2) Upsert durable preferences into human; current goal into task.\n"
        "3) Invalidate superseded facts with block_replace.\n"
        "4) Do not invent secrets; only use transcript evidence.\n"
        "Call tools as needed, then give a short Chinese summary of what you changed."
    )
    return create_agent(get_llm(), BLOCK_LC_TOOLS, system_prompt=system)


def run_online_turn(user_text: str) -> tuple[dict[str, Any], float]:
    """
    Args:
        user_text: 用户输入。

    Returns:
        result: agent 返回值。
        online_ms: 在线耗时毫秒。
    """
    PROD_STATE.recall.add("user", user_text)
    hint = f"{PROD_STATE.core.render_core()}\n\n# User\n{user_text}"
    t0 = time.perf_counter()
    result = build_online_agent().invoke({"messages": [HumanMessage(content=hint)]})
    online_ms = (time.perf_counter() - t0) * 1000
    final = ""
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage) and m.content and not m.tool_calls:
            final = m.content if isinstance(m.content, str) else str(m.content)
            break
    if final:
        PROD_STATE.recall.add("assistant", final)
    return result, online_ms


def run_sleep_consolidate() -> tuple[dict[str, Any], float]:
    """
    Returns:
        result: 闲时 agent 轨迹。
        sleep_ms: 闲时耗时毫秒。
    """
    before = len(PROD_STATE.core.diffs)
    prompt = (
        f"{PROD_STATE.core.render_core()}\n\n"
        f"# Transcript\n{PROD_STATE.recall.transcript()}\n\n"
        "Consolidate memory now."
    )
    t0 = time.perf_counter()
    result = build_sleep_agent().invoke({"messages": [HumanMessage(content=prompt)]})
    sleep_ms = (time.perf_counter() - t0) * 1000
    new_diffs = PROD_STATE.core.diffs[before:]
    note = {
        "sleep_ms": sleep_ms,
        "diffs": [
            {"label": d.label, "old": d.old_version, "new": d.new_version, "patch": d.patch}
            for d in new_diffs
        ],
    }
    PROD_STATE.recall.add("sleep", json.dumps(note, ensure_ascii=False))
    return result, sleep_ms


print(f"LangChain Letta blocks ready | {MODEL}")


## 5. 生产示例：先回复用户，再 sleep consolidate


In [ ]:
def demo_deepseek_memory_blocks() -> None:
    """真实 API：在线先回复 → 闲时再整合；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod_state()

    # 在线：强调低延迟，允许不写块
    r1, online_ms = run_online_turn(
        "我叫 Ada，是 PM，prefers concise。"
        "当前 goal: ship memory-block notebook。"
        "请先简短确认即可；重记忆整理交给闲时，不必现在反复读写块。"
    )
    print("=== online turn ===")
    print(format_agent_messages(r1["messages"]))
    print(f"online_ms={online_ms:.1f}")

    # 人为制造块膨胀，迫使 sleep summarize
    for frag in ["note-a", "note-b", "note-c", "note-d", "note-e", "note-f"]:
        obs = PROD_TOOLS.block_append("human", frag)
        if obs.startswith("Error"):
            break

    before_v = {
        label: PROD_STATE.core.blocks[label].version for label in PROD_STATE.core.blocks
    }
    r2, sleep_ms = run_sleep_consolidate()
    print("\n=== sleep-time consolidate ===")
    print(format_agent_messages(r2["messages"]))
    print(f"sleep_ms={sleep_ms:.1f}")
    print("\n=== core after sleep ===")
    print(PROD_STATE.core.render_core())

    assert online_ms > 0 and sleep_ms > 0
    after_v = {
        label: PROD_STATE.core.blocks[label].version for label in PROD_STATE.core.blocks
    }
    changed = any(after_v[k] > before_v[k] for k in before_v)
    tool_n = count_tool_calls(r2["messages"])
    blob = (
        PROD_STATE.core.blocks["human"].value
        + " "
        + PROD_STATE.core.blocks["task"].value
        + " "
        + format_agent_messages(r2["messages"])
    ).lower()
    assert tool_n >= 1 or changed or "ada" in blob or "concise" in blob
    print("\nPRODUCTION DEMO OK")


demo_deepseek_memory_blocks()
